Author: Daniel Vandevort

Date: June 2026

contact: vandevod@oregonstate.edu

In [ ]:
# imports and setup
import os
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import gc

# pre-processed data
final = r'path/to/lake_ice/data'


In [ ]:
# Define Winter DOY Function
def calculate_winter_doy(date):
    if pd.isna(date):  # Handle NaT values
        return None
    if date.month >= 10:  # From October to December (current year)
        return date.dayofyear - 273  # October 1 = 0 (non-leap year)
    else:  # From January to April (next year)
        return date.dayofyear + 92  # Add days from previous year's winter

## Build the datasets
The `ice_df` dataframe is used for model training and contains only rows where the ice class is observed. That and the rest of the data (temperature) is in the `all_features_df` dataframe. We will use `all_features_df` to create a complete grid of features for all lake-year combinations, which will then be merged with the observed data in `ice_df` to create the final dataset for prediction. This approach allows us to leverage all available temperature data while ensuring that our model is trained only on observed ice conditions.

In [ ]:
# Ensure column names (strings) match your input data. Change as needed.
training_data = []
all_features_data = []
for f in tqdm([i for i in os.listdir(final)], desc='Building dataset...', unit='file'):
    fp = os.path.join(final, f)
    lake_data = pd.read_csv(fp)
    lake_data['date'] = pd.to_datetime(lake_data['date'])
    lake_data = lake_data.sort_values(by='date')
    lake_id = int(f.split('_')[1].split('.')[0])  # Get lake ID

    # Calculate winter DOY and winter year
    lake_data['winter_doy'] = lake_data['date'].apply(calculate_winter_doy) # can only take one input
    lake_data['winter_year'] = lake_data['date'].apply(
        lambda d: d.year if d.month >= 10 else d.year - 1)

    # Create ice binary variable
    lake_data['ice_binary'] = ((lake_data['ice_class'] == 'full') |
                               (lake_data['ice_class'] == 'partial')).astype(int) # this is either 1 or 0 (ice or no ice)

    # Calculate temperature metrics
    lake_data['tmean_C'] = (lake_data['tmin'] + lake_data['tmax']) / 2
    lake_data['freezing_degrees'] = lake_data['tmean_C'].apply(lambda x: max(0 - x, 0))
    lake_data['afdd'] = lake_data.groupby('winter_year')['freezing_degrees'].cumsum()

    # Calculate weekly aggregations per lake on a rolling basis
    lake_data['weekly_mean_low'] = lake_data.groupby('winter_year')['tmin'].transform(
        lambda x: x.rolling(window=7, min_periods=1).mean() # rolling weeks
    )
    lake_data['weekly_afdd'] = lake_data.groupby('winter_year')['afdd'].transform(
        lambda x: x.rolling(window=7, min_periods=1).mean()
    )
    all_features_data.extend(lake_data[[
        'winter_year', 'winter_doy', 'weekly_afdd', 'weekly_mean_low'
    ]].assign(lake_id=lake_id).to_dict('records'))
    # Add only valid rows to training data
    valid_rows = lake_data[~pd.isna(lake_data['ice_class'])]

    training_data.extend(valid_rows[[
        'Elevation_m', 'Latitude', 'weekly_afdd', 'Area_sqkm',
        'weekly_mean_low', 'ice_binary', 'winter_year', 'winter_doy'
    ]].rename(columns={
        'Elevation_m': 'elevation',
        'Latitude': 'latitude',
        'Area_sqkm': 'area',
        'weekly_mean_low':'$ T_{wk,L} $' # If plotting, use Latex to achieve typical subscripting. Otherwise, change name as needed
    }).assign(lake_id=lake_id).to_dict('records'))

# Convert to dataframe and drop NaNs
ice_df = pd.DataFrame(training_data)
ice_df = ice_df.dropna()
all_features_df = pd.DataFrame(all_features_data)
all_features_df.rename(columns={'weekly_mean_low': '$ T_{wk,L} $'}, inplace=True)

In [ ]:
# workspace cleaning. Optional
del(lake_data, valid_rows)
gc.collect

## Create complete grid

In [ ]:
print("\nGenerating complete dataset...")
all_winters = range(ice_df['winter_year'].min(), ice_df['winter_year'].max() + 1)

# Get STATIC lake properties for each lake
lake_properties = ice_df.groupby('lake_id').agg({
    'elevation': 'first',
    'latitude': 'first',
    'area': 'first',
}).reset_index()

# Create a mapping from all features (not just observed)
features_pivot = all_features_df.drop_duplicates(
    subset=['lake_id', 'winter_year', 'winter_doy'], keep='first'
).set_index(['lake_id', 'winter_year', 'winter_doy'])[
    ['weekly_afdd', '$ T_{wk,L} $']
].to_dict('index')

# Create observation mapping (for data_type flag)
obs_pivot = ice_df.drop_duplicates(
    subset=['lake_id', 'winter_year', 'winter_doy'], keep='first'
).set_index(['lake_id', 'winter_year', 'winter_doy'])[
    ['ice_binary']
].to_dict('index')

# Generate complete grid
complete_grid = [
    {
        'lake_id': row['lake_id'],
        'winter_year': year,
        'winter_doy': day,
        'elevation': row['elevation'],
        'latitude': row['latitude'],
        'area': row['area'],
        'weekly_afdd': features_pivot.get((lake_id, year, day), {}).get('weekly_afdd'),  # ← From ALL features
        '$ T_{wk,L} $': features_pivot.get((lake_id, year, day), {}).get('$ T_{wk,L} $'),  # ← From ALL features
        'ice_binary': obs_pivot.get((lake_id, year, day), {}).get('ice_binary'),  # ← From observations only
        'data_type': 'Observed' if (lake_id, year, day) in obs_pivot else 'Modeled',
    }
    for _, row in tqdm(lake_properties.iterrows(), total=len(lake_properties), desc='Generating grid')
    for year in all_winters
    for day in range(1, 272) # this is the ice season range for the study
]
complete_df = pd.DataFrame(complete_grid)
complete_df = complete_df.dropna(subset = ['weekly_afdd']) # removes the 5638 NaNs for afdd and weekly mean low


In [ ]:
# workspace cleaning. Optional
del(all_winters, lake_properties, features_pivot, obs_pivot)
gc.collect()

## Run the Model: training

In [ ]:
#################### THIS IS THE FEATURE SET ###############################
X = ice_df[['elevation', 'latitude', 'area', 'weekly_afdd', '$ T_{wk,L} $']]
############################################################################
y = ice_df['ice_binary'] # Target

In [ ]:
# Use lake_id as the grouping variable for cross-validation
print("=" * 60)
print("Lake-based cross-validation (5-fold)")
print("=" * 60)
group_kfold = GroupKFold(n_splits=5)
cv_scores = []
cv_auc_scores = []
cv_confusion_matrices = []

print("\nPerforming lake-based cross-validation...")
for i, (train_idx, test_idx) in enumerate(group_kfold.split(X, y, groups=ice_df['lake_id'])):
    # Split data
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # Standardize
    scaler1 = StandardScaler()
    X_train_scaled = scaler1.fit_transform(X_train)
    X_test_scaled = scaler1.transform(X_test)
    # Train model
    # default solver lbfgs
    log_reg = LogisticRegression(max_iter=100,
                                 random_state=42,
                                 verbose = 1, # optional
                                 class_weight = 'balanced',
                                 n_jobs = -2) # uses all available cores except 1.
    log_reg.fit(X_train_scaled, y_train)
    # evaluation
    acc = accuracy_score(y_test, log_reg.predict(X_test_scaled))
    auc = roc_auc_score(y_test, log_reg.predict_proba(X_test_scaled)[:, 1])
    cm = confusion_matrix(y_test, log_reg.predict(X_test_scaled))

    cv_scores.append(acc)
    cv_auc_scores.append(auc)
    cv_confusion_matrices.append(cm)
    print(f"Fold {i + 1} split: {len(train_idx)} training samples, {len(test_idx)} testing samples")
    print(f"Percentage split: {len(train_idx) / len(X) * 100:.1f}% train, {len(test_idx) / len(X) * 100:.1f}% test")


print(f"\nCross-validation results:")
print(f"Accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"AUC: {np.mean(cv_auc_scores):.4f} ± {np.std(cv_auc_scores):.4f}")
# Average confusion matrix across folds
avg_cm = np.mean(cv_confusion_matrices, axis=0)
print("\nAverage Confusion Matrix:")
print(avg_cm)


In [ ]:
print("\n" + "=" * 60)
print("Temporal validation (2022+)")
print("=" * 60)

X_time_train = X[ice_df['winter_year'] < 2022]
y_time_train = y[ice_df['winter_year'] < 2022]
X_time_test = X[ice_df['winter_year'] >= 2022]
y_time_test = y[ice_df['winter_year'] >= 2022]

time_scaler = StandardScaler()
X_time_train_scaled = time_scaler.fit_transform(X_time_train)
X_time_test_scaled = time_scaler.transform(X_time_test)

# SAME hyperparameters
time_model = LogisticRegression(max_iter=100,
                                random_state=42,
                                class_weight='balanced',
                                n_jobs = -2)
time_model.fit(X_time_train_scaled, y_time_train)

time_acc = accuracy_score(y_time_test, time_model.predict(X_time_test_scaled))
time_auc = roc_auc_score(y_time_test, time_model.predict_proba(X_time_test_scaled)[:, 1])

print(f"Time validation (2022+ test): Acc = {time_acc:.4f}, AUC = {time_auc:.4f}")

In [ ]:
# workspace cleaning. Optional
del(X_time_train, y_time_train, X_time_test, y_time_test, time_scaler, X_time_train_scaled, X_time_test_scaled, X_time_train_scaled)
gc.collect()

In [ ]:
print("\n" + "=" * 60)
print("Final model")
print("=" * 60)

final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X)

# SAME hyperparameters
lake_ice_model = LogisticRegression(max_iter=100,
                                    random_state=42,
                                    class_weight='balanced',
                                    n_jobs = -2) # default solver = lbfgs
lake_ice_model.fit(X_all_scaled, y)

print("Final model trained on all observations")  # Because we are taking a gap-filling approach
print(f"Training samples: {len(X)}")

## Run the Model: prediction

In [ ]:
# Identify rows needing prediction
prediction_rows = complete_df['ice_binary'].isna()
print(f"Total rows needing prediction: {prediction_rows.sum()}")

# Extract features for all prediction rows
X_to_predict = complete_df.loc[prediction_rows, ['elevation', 'latitude', 'area', 'weekly_afdd', '$ T_{wk,L} $']]

print(f"Predicting ice status for {len(X_to_predict)} rows...")

# Scale and predict using the final model's scaler
X_scaled = final_scaler.transform(X_to_predict)
predictions_proba = lake_ice_model.predict_proba(X_scaled)[:,1]
print(f'length of predictions:',len(predictions_proba))
# Assign predictions
complete_df.loc[prediction_rows, 'ice_probability'] = predictions_proba

# For observed data points, use actual observations
observed_rows = ~prediction_rows
complete_df.loc[observed_rows, 'ice_prediction'] = complete_df.loc[observed_rows, 'ice_binary']
complete_df.loc[observed_rows, 'ice_probability'] = complete_df.loc[observed_rows, 'ice_binary'].astype(float)

# Summary
print(f"\nPrediction Summary:")
print(f"  Observed rows: {observed_rows.sum():6d}")
print(f"  Total rows in complete_df: {len(complete_df):6d}")

In [ ]:
complete_df.drop(columns = 'ice_binary', inplace = True)
# Workspace cleaning (optional)
del(observed_rows, prediction_rows, predictions_proba, X_all_scaled, X_to_predict, X_scaled)
gc.collect()
final_df = pd.DataFrame(complete_df)

## Generate Ice Metrics Dataset


In [ ]:
# Filter for ice days only
ice_metrics = final_df[final_df['ice_probability'] >= 0.5]
calculated_metrics = ice_metrics.groupby(['lake_id', 'winter_year'])['winter_doy'].agg(
    ice_on_doy='min',
    ice_off_doy='max',
    ice_duration='size'
).reset_index()

ice_metrics = ice_metrics.merge(calculated_metrics, on=['lake_id', 'winter_year'], how='left')
ice_metrics['TID'] = ice_metrics['ice_duration'].fillna(0)

print(f"\nIce Metrics Summary:")
print(f"  Total lake-year combinations: {len(ice_metrics)}")
print(f"  Lake-years with ice (>= threshold): {ice_metrics['ice_on_doy'].notna().sum()}")

# Group by lake_id to calculate 10-year variability stats
lake_variability = ice_metrics.groupby('lake_id').agg(
    ice_on_sd=('ice_on_doy', 'std'),
    ice_on_mean=('ice_on_doy', 'mean'),
    ice_off_sd=('ice_off_doy', 'std'),
    ice_off_mean=('ice_off_doy', 'mean'),
    duration_mean=('TID', 'mean'), # TID = total ice days; better than "duration"
    duration_sd=('TID', 'std')
).reset_index()

# Calculate CV for TID (SD / Mean)
lake_variability['TID_cv'] = np.where( # to avoid dividing by zero for lakes that might never freeze
    lake_variability['duration_mean'] > 0,
    lake_variability['duration_sd'] / lake_variability['duration_mean'],
    np.nan  # Leaves NaN for lakes with strictly 0 days of ice
)
ice_metrics = ice_metrics.merge(lake_variability, on=['lake_id'], how='left')

In [ ]:
ice_metrics.to_csv(r'../../Data/IPMs/ice_metrics.csv', index=False) # Update path as needed

In [ ]:
# Save the variability dataframe as its own file. Usefule for visualization
variability_df = ice_metrics[[
    'lake_id', 'winter_year', 'elevation', 'latitude', 'area',
    'ice_on_sd', 'ice_on_mean', 'ice_off_sd', 'ice_off_mean',
    'duration_mean', 'TID_cv'
]].copy()

# Rename the columns as needed
variability_df.rename(columns={
    'winter_year': 'ice_season',
    'duration_mean': 'mean_TID'
}, inplace=True)
variability_summary = variability_df.drop(columns=['ice_season']).drop_duplicates()
variability_summary.to_csv(r'../../Data/IPMs/lake_variability_summary.csv', index=False) # update path as needed